In [30]:
import os
import boto3
import requests
from io import BytesIO
import yaml



In [31]:
#read yaml file
with open('credentials.yml', 'r') as f:
    credentials = yaml.safe_load(f)
    aws_access_key_id = credentials.get('aws', {}).get('aws_access_key_id')
    aws_secret_access_key = credentials.get('aws', {}).get('aws_secret_access_key')


In [32]:

#create an authorised session using boto3 + credentials
session = boto3.session.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)
# create s3 client to interact with AWS S3
s3 = session.client('s3')

dest_s3_bucket = "cm-aws-s3-destination"
dest_s3_path_prefix = "k1/nhi/gtfs"

SA Metro GTFS
- Loading Mode: Initial Full Load + Incremental Load
    - Initial Full Load: Load the previous 10 versions of GTFS
    - Incremental Load:
        - Get the latest version in `landing zone` & latest version available from API.
        - Execute load if the API's latest version has not been ingested.  
- Partitioned Destination: `<landing_bucket>/gtfs/<FEED VERSION>`

In [33]:
# # FULL LOAD
# # 1. read data from SA Metro GTFS feed
# # 2. upload feed version into landing bucket
# base_url = "http://gtfs.adelaidemetro.com.au/v1"
# latest_version_number = "static/latest/version.txt"
# latest_version_feed = "static/latest/google_transit.zip"
# latest_version_feed_template = "static/{version}/google_transit.zip"

# def extract_gtfs_to_s3(feed_version: int) -> str:
#      feed_version_url = f"{base_url}/{latest_version_feed_template.format(version=feed_version)}"


#     try:
#         resp = requests.get(feed_version_url)
#         if resp.ok:
#             s3_key = f"{s3_path_prefix}/{feed_version}_google_transit.zip"
#             s3_upload_fileobj(BytesIO(resp.content), Bucket=s3_bucket, Key=s3_key)
#             return f"SUCCESS: Version {feed_version}"
#         else:
#             return f"FAILED (HTTP {resp.status_code}): Version {feed_version}"
#     except Exception as e:
#         return f"ERROR: Version {feed_version} - {str(e)}"

base_url = "http://gtfs.adelaidemetro.com.au/v1"
latest_version_number = "static/latest/version.txt"
latest_version_feed_template = "static/{version}/google_transit.zip"

def extract_gtfs_to_s3(feed_version: int) -> str:
    feed_version_url = f"{base_url}/{latest_version_feed_template.format(version=feed_version)}"

    try:
        resp = requests.get(feed_version_url, timeout=30)

        if not resp.ok:
            return f"FAILED (HTTP {resp.status_code}): Version {feed_version}"

        s3_key = f"{s3_path_prefix}/{feed_version}_google_transit.zip"

        # Upload bytes to S3
        fileobj = BytesIO(resp.content)
        s3.upload_fileobj(Fileobj=fileobj, Bucket=s3_bucket, Key=s3_key)

        return f"SUCCESS: Version {feed_version}"

    except Exception as e:
        return f"ERROR: Version {feed_version} - {str(e)}"


In [34]:
# INCREMENTAL LOAD
# 1. Get: API's latest version & AWS S3's versions
# 2. Compare & logging whether the latest version is already ingested
# 3. Execute the ingestion if not already ingested

# Practice: define script into tasks / functions
# e.g., Func1: Get API Latest version
# Func2: Get AWS S3 versions and compare if API latest version is already ingested -> True/False
# Func3: Execute the API latest version ingestion if Func2 return False

latest_version_number_url = f"{base_url}/{latest_version_number}"
resp = requests.get(latest_version_number_url)
latest_version = int(resp.text)
print(f'latest available in source: {latest_version}')

latest available in source: 1626


In [35]:
#Simple sequential
old_versions = range(1600, latest_version)
#print(old_versions)
for feed_version in old_versions:
    result = extract_gtfs_to_s3(feed_version=feed_version)
    print(result)

SUCCESS: Version 1600
SUCCESS: Version 1601
SUCCESS: Version 1602
SUCCESS: Version 1603
SUCCESS: Version 1604
SUCCESS: Version 1605
SUCCESS: Version 1606
SUCCESS: Version 1607
SUCCESS: Version 1608
SUCCESS: Version 1609
SUCCESS: Version 1610
SUCCESS: Version 1611
SUCCESS: Version 1612
SUCCESS: Version 1613
SUCCESS: Version 1614
FAILED (HTTP 403): Version 1615
SUCCESS: Version 1616
FAILED (HTTP 403): Version 1617
SUCCESS: Version 1618
SUCCESS: Version 1619
SUCCESS: Version 1620
SUCCESS: Version 1621
SUCCESS: Version 1622
SUCCESS: Version 1623
SUCCESS: Version 1624
SUCCESS: Version 1625


In [37]:

#dest_s3_bucket = "cm-aws-s3-destination"
#dest_s3_path_prefix = "k1/nhi/gtfs"

resp = s3.list_objects_v2(Bucket=dest_s3_bucket, Prefix=dest_s3_path_prefix)

obj_keys = [obj["Key"] for obj in resp.get("Contents", [])]
obj_keys

['k1/nhi/gtfs/1600_google_transit.zip',
 'k1/nhi/gtfs/1601_google_transit.zip',
 'k1/nhi/gtfs/1602_google_transit.zip',
 'k1/nhi/gtfs/1603_google_transit.zip',
 'k1/nhi/gtfs/1604_google_transit.zip',
 'k1/nhi/gtfs/1605_google_transit.zip',
 'k1/nhi/gtfs/1606_google_transit.zip',
 'k1/nhi/gtfs/1607_google_transit.zip',
 'k1/nhi/gtfs/1608_google_transit.zip',
 'k1/nhi/gtfs/1609_google_transit.zip',
 'k1/nhi/gtfs/1610_google_transit.zip',
 'k1/nhi/gtfs/1611_google_transit.zip',
 'k1/nhi/gtfs/1612_google_transit.zip',
 'k1/nhi/gtfs/1613_google_transit.zip',
 'k1/nhi/gtfs/1614_google_transit.zip',
 'k1/nhi/gtfs/1616_google_transit.zip',
 'k1/nhi/gtfs/1618_google_transit.zip',
 'k1/nhi/gtfs/1619_google_transit.zip',
 'k1/nhi/gtfs/1620_google_transit.zip',
 'k1/nhi/gtfs/1621_google_transit.zip',
 'k1/nhi/gtfs/1622_google_transit.zip',
 'k1/nhi/gtfs/1623_google_transit.zip',
 'k1/nhi/gtfs/1624_google_transit.zip',
 'k1/nhi/gtfs/1625_google_transit.zip']